# Percentils, Bootstrapping and Confidence Intervals

This notebook reviews Lecture 23: Confidence Intervals and Lecture 24: Interpreting Confidence. 

**Helpful Resource:**
- [Python Reference](http://data8.org/sp22/python-reference.html): Cheat sheet of helpful array & table methods used in Data 8!

**Recommended Readings**: 
- [Percentiles](https://inferentialthinking.com/chapters/13/1/Percentiles.html)
- [Bootstrap](https://inferentialthinking.com/chapters/13/2/Bootstrap.html)
- [Confidence Intervals](https://inferentialthinking.com/chapters/13/3/Confidence_Intervals.html)

In [ ]:
from datascience import *
%matplotlib inline
path_data = '../../../assets/data/'
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')
import numpy as np
from math import ceil

### Percentiles

Use the student data to identify the 20th, 40th, 60th, and 80th quartile of the height. Calculate your answers in two ways: once using the `percentile` function provided by the `datascience` package, and a second time by using the definition of percentiles. The position $L$ in a sorted list for the k-th percentile is given by: 

$L = \frac{k}{100} * n$

If L is not an integer, you can use the `ceil` function in the `math` package to round $L$ up to the next greater integer. 

In [ ]:
students = Table.read_table('./data/student_data.csv')
hgt = students.column('HEIGHT')
hgt
hgt.sort()
hgt
#students

#hgt.item(ceil(20/100 * len(hgt)) - 1), hgt.item(ceil(40/100 * len(hgt)) - 1), hgt.item(ceil(60/100 * len(hgt)) - 1), hgt.item(ceil(80/100 * len(hgt)) - 1)




The `boxplot` shows some of the important percentiles from a given set of data. Run the following code to examine the boxplot of the student height data. How do your percentiles compare with these? 

In [ ]:
students.select('HEIGHT').boxplot()

### Bootstrapping

Prices of 10 textbooks are shown below. 

In [ ]:
ten_books = make_array(35, 83, 102, 58, 91, 155, 205, 220, 120, 67, 99)

Find the sample mean. 

In [ ]:
np.average(ten_books)

Use `sample` to create a bootstrap sample. Does your sample contain any repeated values? Is this a problem? 

In [ ]:
book_table = Table().with_column('Books', ten_books)
bootstrap_books=  book_table.sample().column('Books')
bootstrap_books

Find the mean of this bootstrap sample. 

In [ ]:
np.average(bootstrap_books)

If we generate many means from all the bootstrap samples, where will this bootstrap distribution be centered?  

In [ ]:
bootstrap_means = make_array()
for i in np.arange(10000):
    bootstrap_books=  book_table.sample().column('Books')
    one_bootstrap_mean = np.average(bootstrap_books)
    bootstrap_means = np.append(bootstrap_means, one_bootstrap_mean)
Table().with_column('Bootstrap means', bootstrap_means).hist()

In [ ]:
np.average(bootstrap_means), np.average(ten_books)
percentile(2.5, bootstrap_means), percentile(97.5, bootstrap_means)

In [ ]:
voters = make_array(0, 1, 1, 0, 1, 1, 0, 1, 1, 1)
t = Table().with_column('Vote', voters)
bootstrap_prop = make_array()
for i in np.arange(10000):
    bootstrap_prop = np.append(bootstrap_prop, np.average(t.sample().column('Vote')))
Table().with_column("bootstrap proportions", bootstrap_prop).hist()
np.average(bootstrap_prop)

### Confidence Intervals from Previous Simulations

What percentage of Americans believe in global warming?  A survey on 2,251 randomly selected individuals conducted in October 2020 found that 1,328 answered Yes to the question “Is there solid evidence of global warming?”   A bootstrap distribution for this data is shown.  

<img src="./img/bootstrap_proportion.png" width=600>

What does each value represent in this graph? 

a bootstrap proportion obtained from the original sample of n=12251 by sampling with replacement

Use the information in the graph to give and interpret a 95% CI for the proportion of Americans who believe there is solid evidence of global warming. (Note: you can use the closest values on the X-axis to approximate the middle 95% of the total area) 

approximately from 0.57 to 0.61

A survey of 50 four-year colleges in the U.S. shows their tuition in 2020. Based on this sample, bootstrapping was used to resample from the 50 colleges and the distribution of the bootstrap sample means is shown below. 
<img src="./img/bootstrap_mean.GIF" width=800>

Use this graph to derive the 95% confidence interval of the mean tuition cost in U.S. 4-year colleges. 


approx from 29000 to 36000



### Constructing a Confidence Interval for a Single Mean

The following code uses the student data to create a bootstrap confidence interval for the average age. In the usage of ```table.sample```, is it doing sampling with or without replacement?  What would happen otherwise?   

In [ ]:
students
np.average(students.column('AGE'))
students.hist('AGE')

In [ ]:
bootstrap_means = make_array()
for i in np.arange(50000):
    resampled_mean = np.average(students.sample().column('AGE'))
    bootstrap_means = np.append(bootstrap_means, resampled_mean)

Table().with_column('bootstrap means', bootstrap_means).hist()

Use the `bootstrap_means` array and `percentile` function to compute the 90%, 95% and 99% confidence intervals for the average age. 

In [ ]:
bootstrap_means.sort()
print(bootstrap_means.item(ceil(0.05*len(bootstrap_means))+1), bootstrap_means.item(ceil(0.95*len(bootstrap_means))+1))
# you can draw the interval on the histogram by using plots.plot([left, right], [0, 0], color='red', lw=8);
# 90% confidence interval
percentile(5, bootstrap_means), percentile(95, bootstrap_means)

# 95% C.I.
print("95% confidence interval", percentile(2.5, bootstrap_means), percentile(97.5, bootstrap_means))

# 99% C.I.
print("99% confidence interval", percentile(0.5, bootstrap_means), percentile(99.5, bootstrap_means))

np.std(students.column('AGE'))

### Confidence Interval for a Difference of Two Means

Can you modify the simulation above to use bootstrapping to construct a confidence interval between the average height of male and female students? (Note: you can use ```average_male - average_female``` as the bootstrap statistic)

In [ ]:
students.group('SEX', np.average)
#students.group('SEX')

In [ ]:
bootstrap_diff_means = make_array()
males = students.where('SEX', 'Male').column('HEIGHT')
females = students.where('SEX', 'Female').column('HEIGHT')

for i in np.arange(10000):
    mean_male = np.average(np.random.choice(males, 33))
    mean_female = np.average(np.random.choice(females, 41))
    bootstrap_diff_means = np.append(bootstrap_diff_means, mean_male - mean_female)

Table().with_column('bootstrap means', bootstrap_diff_means).hist()

In [ ]:
print('95% confidence interval', percentile(2.5, bootstrap_diff_means), percentile(97.5, bootstrap_diff_means))
# the interval only contains positive numbers, so the average height of males is indeed higher than that of females

Using the NBA data to construct a 95% confidence interval for the average salary of players in 2020 and compare it with the 2010 season.  

In [ ]:
nba = Table.read_table('./data/nba_salaries.csv')
nba20 = nba.where('season', are.equal_to(2020))
nba10 = nba.where('season', are.equal_to(2010))
nba20.num_rows, nba10.num_rows

In [ ]:
def average_salary_bootstrap(table):
    bootstrap_means = make_array()
    for i in np.arange(5000):
        resampled_mean = np.average(table.sample().column('salary'))
        bootstrap_means = np.append(bootstrap_means, resampled_mean)
    return bootstrap_means

means20 = average_salary_bootstrap(nba20)
means10 = average_salary_bootstrap(nba10)

In [ ]:
print('Year 2020: ', percentile(2.5, means20), percentile(97.5, means20))
print('Year 2010: ', percentile(2.5, means10), percentile(97.5, means10))

Use the NBA data, can you find how many percent of all the players play the Center (C) position in 2020 v.s. 2010?

In [ ]:
np.count_nonzero(nba20.column('position')=='C'), np.count_nonzero(nba10.column('position')=='C')

In [ ]:
def center_percent_bootstrap(table):
    bootstrap_props = make_array()
    for i in np.arange(5000):
        resampled_prop = np.count_nonzero(table.sample().column('position')=='C') / table.num_rows
        bootstrap_props = np.append(bootstrap_props, resampled_prop)
    return bootstrap_props

centers20 = center_percent_bootstrap(nba20)
centers10 = center_percent_bootstrap(nba10)
print('Year 2020: ', percentile(2.5, centers20), percentile(97.5, centers20))
print('Year 2010: ', percentile(2.5, centers10), percentile(97.5, centers10))


Since there is no separation between the intervals, we cannot say that the percentage of centers has declined in 2020 as compared to 2010. 

In [ ]:
print('Year 2020: ', percentile(15, centers20), percentile(85, centers20))
print('Year 2010: ', percentile(15, centers10), percentile(85, centers10))

### Confidence Interval for Proportions 

The sample proportions can be seen as the mean of a binary variable, where $X=1$ represents the heads of a coin toss, and $X=0$ otherwise. This variable is automatically created when you use ```np.count_nonzero```. For example, to construct a confidence interval for the proportion of students who use social media, we use the following code to generate the bootstrap distribution. 

In [ ]:
bootstrap_props = make_array()
for i in np.arange(5000):
    resampled_prop = np.count_nonzero(students.sample().column('SOCIAL') == 'Yes') / students.num_rows
    bootstrap_props = np.append(bootstrap_props, resampled_prop)
Table().with_column('sample props', bootstrap_props).hist()

In [ ]:
percentile(2.5, bootstrap_props), percentile(97.5, bootstrap_props)

Use the NBA data to produce 95% confidence intervals for the proportion of centers in 2020 v.s. 2010.

In [ ]:
def prop_center_bootstrap(table):
    bootstrap_props = make_array()
    for i in np.arange(5000):
        resampled_prop = np.count_nonzero(table.sample().column('position') == 'C') / table.num_rows
        bootstrap_props = np.append(bootstrap_props, resampled_prop)
    return bootstrap_props


Based on these confidence intervals, is there an evidence that the percentage of centers in NBA changed significantly from 2010 to 2020? 

Here is a recent survey from Pew Research with regard to how Americans view the military action in Iran. The description of the methodology can be found [here](https://www.pewresearch.org/politics/2026/03/25/iran-methodology/). Construct a 99% bootstrapping confidence interval of the percentage of Americans who disapprove of the president's handling of the conflict. 

In [ ]:
# Data from Pew Research Center's March 2026 survey of 3,524 Americans
# About 6-in-10 Americans (61%) disapprove the president's handling of the conflict, while 37% approve.

iran_data = Table().with_column("Disapprove", np.concatenate([np.ones(round(3524*0.61)), np.zeros(round(3524*0.39))]))
def prop_oppose_bootstrap(table):
    bootstrap_props = make_array()
    for i in np.arange(1000):
        resampled_prop = np.count_nonzero(table.sample().column('Disapprove')) / table.num_rows
        bootstrap_props = np.append(bootstrap_props, resampled_prop)
    return bootstrap_props

bootstrap_disapprove_props = prop_oppose_bootstrap(iran_data)
Table().with_column('Bootstrap disapprove props', bootstrap_disapprove_props).hist()
percentile(0.5, bootstrap_disapprove_props), percentile(99.5, bootstrap_disapprove_props)